# HW5. Три эксперимента с видимыми шагами

Каждый эксперимент ниже проходит отдельно через модель → LoRA → Trainer → train() → сохранение. Параметры заданы непосредственно в notebook; файлы configs/rl_*.json и запуск argparse здесь не используются.

Обучаем 120 шагов optimizer. Число шагов задано в `COMMON_GRPO["max_steps"]`: при положительном `max_steps` Trainer останавливается на нём и не доходит эпоху. Выбранный checkpoint — шаг 120. На этом же шаге один раз оценивается вся подготовленная validation. Затем evaluate_all.sh оценит baseline и три модели через Docker. Для CPU-анализа сохранённых результатов есть analysis.ipynb. Разбор данных до обучения — в eda.ipynb.

Перед первым запуском перезапусти kernel. Дальше выполняй сверху вниз. Пока идёт подготовка данных, загрузка весов, обучение или сохранение, соответствующая ячейка остаётся выполняющейся. Длинный шаг генерации не обязан сразу продвигать общий счётчик optimizer-шагов.

## 1. GPU — до импорта PyTorch

На dev-машине GPU=1 выбирает вторую физическую карту. В Docker, запущенном с --gpus device=1, используй GPU=0: внутри видна единственная карта с индексом 0. Изменение переменной после инициализации CUDA требует перезапуска kernel.

In [1]:
import os

GPU = '1'  # В Docker с одной выделенной GPU поставь '0'.
os.environ['CUDA_VISIBLE_DEVICES'] = GPU

In [2]:
import gc
import json
import time
from importlib.metadata import version
from pathlib import Path
from pprint import pprint

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import set_seed
from trl import GRPOConfig, GRPOTrainer

from prepare import prepare_data, load_examples
from train import (
    MODEL_ID, MODEL_REVISION, load_tokenizer, build_training_rows,
    load_base_model, make_training_reward, save_training_results,
)

assert torch.cuda.device_count() == 1, 'Нужна одна видимая GPU. Перезапусти kernel после выбора.'
print('GPU:', torch.cuda.get_device_name(0))
print('Память GPU, GiB:', torch.cuda.get_device_properties(0).total_memory / 1024**3)

/home/edu/amsafin/HSE-Avito/LLM/hw5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA A100-SXM4-80GB
Память GPU, GiB: 79.251708984375


## 2. Данные: реальные диалоги и отдельные эталоны

Сначала разбираем весь ToolACE и разделяем целые диалоги. Затем строим примеры (prompt, answer, tools). Этот этап не использует GPU. Причины исключения исходных диалогов сохраняются в preparation.json.

In [3]:
print('Разбираем ToolACE и готовим train/validation...', flush=True)
report = prepare_data('data/raw/data.json', 'data/experiments')
train_examples = load_examples('data/experiments/train.jsonl')
validation_examples = load_examples('data/experiments/validation.jsonl')
print('Принято train-примеров:', len(train_examples))
print('Принято validation-примеров:', len(validation_examples))
print('Исключено исходных диалогов:', len(report['excluded_dialogues']))
print('Первый эталон:')
pprint(train_examples[0][1])

Разбираем ToolACE и готовим train/validation...
Принято train-примеров: 11824
Принято validation-примеров: 1292
Исключено исходных диалогов: 659
Первый эталон:
{'content': '',
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': {'evaluation_criteria': [{'criterion': 'water '
                                                                                  'quality',
                                                                     'measurement_method': 'spectral '
                                                                                           'analysis'},
                                                                    {'criterion': 'soil '
                                                                                  'nutrients',
                                                                     'measurement_method': 'chemical '
                                                                                           'testing'}],
                   

## 3. Токенизатор и Dataset

Здесь виден прогресс Render/tokenize для каждой части. Все примеры проходят через настоящий шаблон Qwen. Prompt длиннее 3072 токенов исключается целиком с подсчётом. Эталон и инструменты хранятся в JSON-колонках только для функции награды; во вход модели входит только отрендеренный prompt.

In [4]:
tokenizer = load_tokenizer()
MAX_PROMPT_TOKENS = 3072
print('Train:', flush=True)
train_rows, skipped_train = build_training_rows(train_examples, tokenizer, MAX_PROMPT_TOKENS)
print('Validation:', flush=True)
validation_rows, skipped_validation = build_training_rows(validation_examples, tokenizer, MAX_PROMPT_TOKENS)
for row in train_rows:
    row['split'] = 'train'
for row in validation_rows:
    row['split'] = 'validation'
train_dataset = Dataset.from_list(train_rows)
validation_dataset = Dataset.from_list(validation_rows)
print(train_dataset)
print(validation_dataset)
print('Исключено по длине train / validation:', skipped_train, skipped_validation)
print('Конец первого входа модели:')
print(train_rows[0]['prompt'][-500:])
print('Отдельный эталон:')
pprint(json.loads(train_rows[0]['answer_json']))

Train:


Render/tokenize: 100%|██████████| 11824/11824 [00:19<00:00, 608.42example/s]

Validation:



Render/tokenize: 100%|██████████| 1292/1292 [00:02<00:00, 585.72example/s]


Dataset({
    features: ['prompt', 'answer_json', 'tools_json', 'example_index', 'split'],
    num_rows: 11808
})
Dataset({
    features: ['prompt', 'answer_json', 'tools_json', 'example_index', 'split'],
    num_rows: 1291
})
Исключено по длине train / validation: 16 1
Конец первого входа модели:
><|im_end|>
<|im_start|>user
As part of our conservation efforts, we're focusing on the cheetah habitat in the Savannah region. We need to evaluate the habitat health during spring and, based on that, track the movements of the Cheetah, Acinonyx jubatus, for the following 30 days. The evaluation criteria should include water quality, measured via spectral analysis, and soil nutrients, assessed through chemical testing. Could you help with that?<|im_end|>
<|im_start|>assistant
<think>

</think>


Отдельный эталон:
{'content': '',
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': {'evaluation_criteria': [{'criterion': 'water '
                                                     

## 4. Общие настройки

Для A100 80 GB шаг такой же, как у короткого прогона: 8 ответов и 8 генераций одного промпта. Train и eval batch одинаковые. Максимум ответа — 256 токенов.

Число шагов — `max_steps` в ячейке ниже. `num_train_epochs` при этом не используется. В TRL `per_device_train_batch_size` считает ответы, поэтому промптов на шаг равно `per_device_train_batch_size / num_generations`. При батче 8 и восьми генерациях это один промпт на шаг, за 120 шагов — 120 промптов.

Потолок ответа — `max_completion_length=256`. По `eda.ipynb` эталон после промпта имеет медиану 68 токенов, p90 216, p99 435. Лимит 192 обрезает 13% эталонов, 256 — 6.4%. Берём 256: это покрывает типичный вызов и не держит генерацию до 1024.

Генерация идёт через vLLM в том же процессе: `use_vllm=True`, `vllm_mode="colocate"`. Под кэш vLLM отдано 30% памяти GPU. `vllm_max_model_length=3328` — это потолок промпта 3072 плюс потолок ответа 256. Модель для этого загружается полностью, `load_base_model(language_only=False)`: vLLM читает тот же чекпоинт и сверяет веса по именам `model.language_model.*`. Текстовая модель такие имена не отдаёт.

`gradient_checkpointing` выключен. На шаге 8 коротких ответов активации помещаются рядом с копией vLLM, и backward не пересчитывает forward.

disable_tqdm=False включает progress bar Trainer. logging_steps=1 выводит метрики каждого optimizer-шага. `eval_strategy="steps"` и `eval_steps=120` запускают validation один раз, внутри того же `train()`, когда счётчик доходит до 120. До конца эпохи обучение не доходит, поэтому `eval_strategy="epoch"` эту проверку бы пропустил. Validation использует sampling TRL с восемью ответами на запрос; итоговый BFCL отдельно использует фиксированную greedy-генерацию.

In [5]:
COMMON_GRPO = {
    'num_train_epochs': 1.0,
    'max_steps': 120,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 8,
    'gradient_accumulation_steps': 1,
    'num_generations': 8,
    'max_completion_length': 256,
    'temperature': 1.0,
    'beta': 0.0,
    'loss_type': 'grpo',
    'scale_rewards': 'group',
    'bf16': True,
    'gradient_checkpointing': False,
    'mask_truncated_completions': True,
    'use_vllm': True,
    'vllm_mode': 'colocate',
    'vllm_gpu_memory_utilization': 0.3,
    'vllm_max_model_length': 3328,
    'lr_scheduler_type': 'constant',
    'weight_decay': 0.0,
    'eval_strategy': 'no',
    'save_strategy': 'no',
    'logging_steps': 1,
    'logging_first_step': True,
    'disable_tqdm': False,
    'report_to': 'none',
    'remove_unused_columns': False,
    'seed': 42,
}
pprint(COMMON_GRPO)

{'beta': 0.0,
 'bf16': True,
 'disable_tqdm': False,
 'eval_strategy': 'no',
 'gradient_accumulation_steps': 1,
 'gradient_checkpointing': False,
 'logging_first_step': True,
 'logging_steps': 1,
 'loss_type': 'grpo',
 'lr_scheduler_type': 'constant',
 'mask_truncated_completions': True,
 'max_completion_length': 256,
 'max_steps': 120,
 'num_generations': 8,
 'num_train_epochs': 1.0,
 'per_device_eval_batch_size': 8,
 'per_device_train_batch_size': 8,
 'remove_unused_columns': False,
 'report_to': 'none',
 'save_strategy': 'no',
 'scale_rewards': 'group',
 'seed': 42,
 'temperature': 1.0,
 'use_vllm': True,
 'vllm_gpu_memory_utilization': 0.3,
 'vllm_max_model_length': 3328,
 'vllm_mode': 'colocate',
 'weight_decay': 0.0}


## RL-1: точная награда

Проверяем обучение по полному совпадению вызовов и аргументов.

Выполни все ячейки этого блока, включая сохранение и освобождение памяти, прежде чем переходить к следующему эксперименту. Выходные папки должны быть пустыми: предыдущие результаты не перезаписываются.

In [6]:
run_name = 'rl_exact'
learning_rate = 2e-05
reward_weights = {'exact': 1.0, 'functions': 0.0, 'format': 0.0}
output_dir = Path('results') / run_name
model_dir = Path('checkpoints') / run_name / 'final'
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(f'Результаты уже существуют: {output_dir}')
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(42)
print('Эксперимент:', run_name)
print('Learning rate:', learning_rate)
print('Награда:', reward_weights)

Эксперимент: rl_exact
Learning rate: 2e-05
Награда: {'exact': 1.0, 'functions': 0.0, 'format': 0.0}


### Загрузить исходную модель и подключить LoRA

Веса каждый раз начинаются с одной и той же исходной ревизии. Ниже видны LoRA-параметры, число обучаемых весов и фактически занятая GPU-память.

In [7]:
print('Загрузка исходной Qwen...', flush=True)
gpu_memory_before = torch.cuda.mem_get_info()
torch.cuda.reset_peak_memory_stats()
model = load_base_model(language_only=False).to('cuda')
lora_config = LoraConfig(
    task_type='CAUSAL_LM',
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Загрузка исходной Qwen...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 617/617 [00:00<00:00, 6031.68it/s]


trainable params: 417,792 || all params: 2,213,659,456 || trainable%: 0.0189
Выделено GPU-памяти, GiB: 4.1240620613098145


### Создать Trainer

Функция награды получает реальные генерации, разбирает их и сравнивает с отдельным эталоном. Все генерации и компоненты награды сохраняются в results/<имя>/. Training и validation пишутся в разные файлы.

In [ ]:
eos_token_ids = model.generation_config.eos_token_id or []
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
eos_token_ids = set(eos_token_ids) | {tokenizer.eos_token_id}
training_reward = make_training_reward(
    output_dir=output_dir,
    weights=reward_weights,
    eos_token_ids=eos_token_ids,
    max_completion_length=COMMON_GRPO['max_completion_length'],
    num_generations=COMMON_GRPO['num_generations'],
)
training_args = GRPOConfig(
    output_dir=str(output_dir),
    learning_rate=learning_rate,
    **COMMON_GRPO,
)
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=training_reward,
    train_dataset=train_dataset,
    eval_dataset=None,
    processing_class=tokenizer,
)
settings = {
    **COMMON_GRPO,
    'learning_rate': learning_rate,
    'reward_weights': reward_weights,
    'model': MODEL_ID,
    'revision': MODEL_REVISION,
    'max_prompt_tokens': MAX_PROMPT_TOKENS,
    'lora_rank': 8,
    'lora_alpha': 16,
    'target_modules': ['q_proj', 'v_proj'],
    'training_examples': len(train_dataset),
    'validation_examples': 0,
    'skipped_long_prompts': skipped_train,
    'skipped_long_validation_prompts': skipped_validation,
    'checkpoint_selection': 'optimizer step 120',
    'versions': {name: version(name) for name in ['torch', 'transformers', 'trl', 'peft', 'datasets']},
}
(output_dir / 'config.json').write_text(json.dumps(settings, indent=2))
print('Trainer готов. Следующая ячейка запускает обучение.')

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.35it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.35it/s]

Ca

Trainer готов. Следующая ячейка запускает обучение.


### Обучение и validation

Это прямой trainer.train() в текущем kernel. Progress bar показывает optimizer-шаги и останавливается на 120. Подготовка группы генераций может занимать десятки секунд до первого обновления счётчика. Когда счётчик дойдёт до 120, внутри того же вызова начнётся validation.

In [ ]:
initial_parameters = {}
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        initial_parameters[name] = parameter.detach().float().cpu().clone()
torch.cuda.synchronize()
start = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
train_seconds = time.perf_counter() - start
print('Обучение и validation завершены, секунд:', round(train_seconds, 1))
pprint(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Autotuning kernel l2norm_fwd_kernel with config BT: 8, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 16, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 32, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 64, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 128, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 8, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 16, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 32, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 64, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg

Step,Training Loss,Validation Loss


Autotuning kernel l2norm_fwd_kernel with config BT: 8, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 16, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 32, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 64, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 128, num_warps: 1, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 8, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 16, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 32, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel l2norm_fwd_kernel with config BT: 64, num_warps: 2, num_ctas: 1, num_stages: 3, maxnreg

### Сохранить результаты и полную модель

Сначала сохраняем адаптер, историю, summary и validation-метрики. Затем объединяем LoRA с уже загруженной моделью и сохраняем полные веса для evaluate.py. После этого освобождаем память для следующего запуска.

In [11]:
print('Сохраняем адаптер и метрики...', flush=True)
if 'train_seconds' not in globals():
    train_seconds = None
summary = save_training_results(
    trainer=trainer,
    output_dir=output_dir,
    initial_parameters=initial_parameters,
    train_seconds=train_seconds,
    gpu_memory_before=gpu_memory_before,
)
pprint(summary)
print('Сохраняем объединённую модель:', model_dir, flush=True)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
del trainer, model, merged_model, initial_parameters
gc.collect()
torch.cuda.empty_cache()
print('Готово:', run_name)
print('Осталось выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Сохраняем адаптер и метрики...
{'changed_parameter_tensors': 24,
 'epoch': 0.01016260162601626,
 'finite_training_metrics': True,
 'generated_completions': 960,
 'gpu': 'NVIDIA A100-SXM4-80GB',
 'gpu_free_before_load_gib': 78.838623046875,
 'gpu_total_gib': 79.251708984375,
 'groups_with_reward_variation': 44,
 'optimizer_steps': 120,
 'parameter_delta_l2': 0.14889014083390464,
 'peak_allocated_gib': 52.10128355026245,
 'peak_reserved_gib': 77.47265625,
 'reward_groups': 120,
 'status': 'updated',
 'train_seconds': None,
 'trainable_parameters': 417792,
 'truncated_completions': 86}
Сохраняем объединённую модель: checkpoints/rl_exact/final


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.59s/it]


Готово: rl_exact
Осталось выделено GPU-памяти, GiB: 26.136975288391113


## RL-2: составная награда

Меняем только награду относительно RL-1. Гипотеза: частичный сигнал облегчит обучение.

Выполни все ячейки этого блока, включая сохранение и освобождение памяти, прежде чем переходить к следующему эксперименту. Выходные папки должны быть пустыми: предыдущие результаты не перезаписываются.

In [12]:
run_name = 'rl_shaped'
learning_rate = 2e-05
reward_weights = {'exact': 1.0, 'functions': 0.2, 'format': 0.05}
output_dir = Path('results') / run_name
model_dir = Path('checkpoints') / run_name / 'final'
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(f'Результаты уже существуют: {output_dir}')
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(42)
print('Эксперимент:', run_name)
print('Learning rate:', learning_rate)
print('Награда:', reward_weights)

Эксперимент: rl_shaped
Learning rate: 2e-05
Награда: {'exact': 1.0, 'functions': 0.2, 'format': 0.05}


### Загрузить исходную модель и подключить LoRA

Веса каждый раз начинаются с одной и той же исходной ревизии. Ниже видны LoRA-параметры, число обучаемых весов и фактически занятая GPU-память.

In [13]:
print('Загрузка исходной Qwen...', flush=True)
gpu_memory_before = torch.cuda.mem_get_info()
torch.cuda.reset_peak_memory_stats()
model = load_base_model(language_only=False).to('cuda')
lora_config = LoraConfig(
    task_type='CAUSAL_LM',
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Загрузка исходной Qwen...


Loading weights: 100%|██████████| 617/617 [00:00<00:00, 6210.44it/s]


trainable params: 417,792 || all params: 2,213,659,456 || trainable%: 0.0189
Выделено GPU-памяти, GiB: 30.262425899505615


### Создать Trainer

Функция награды получает реальные генерации, разбирает их и сравнивает с отдельным эталоном. Все генерации и компоненты награды сохраняются в results/<имя>/. Training и validation пишутся в разные файлы.

In [16]:
eos_token_ids = model.generation_config.eos_token_id or []
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
eos_token_ids = set(eos_token_ids) | {tokenizer.eos_token_id}
training_reward = make_training_reward(
    output_dir=output_dir,
    weights=reward_weights,
    eos_token_ids=eos_token_ids,
    max_completion_length=COMMON_GRPO['max_completion_length'],
    num_generations=COMMON_GRPO['num_generations'],
)
training_args = GRPOConfig(
    output_dir=str(output_dir),
    learning_rate=learning_rate,
    **COMMON_GRPO,
)
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=training_reward,
    train_dataset=train_dataset,
    eval_dataset=None,
    processing_class=tokenizer,
)
settings = {
    **COMMON_GRPO,
    'learning_rate': learning_rate,
    'reward_weights': reward_weights,
    'model': MODEL_ID,
    'revision': MODEL_REVISION,
    'max_prompt_tokens': MAX_PROMPT_TOKENS,
    'lora_rank': 8,
    'lora_alpha': 16,
    'target_modules': ['q_proj', 'v_proj'],
    'training_examples': len(train_dataset),
    'validation_examples': 0,
    'skipped_long_prompts': skipped_train,
    'skipped_long_validation_prompts': skipped_validation,
    'checkpoint_selection': 'optimizer step 120',
    'versions': {name: version(name) for name in ['torch', 'transformers', 'trl', 'peft', 'datasets']},
}
(output_dir / 'config.json').write_text(json.dumps(settings, indent=2))
print('Trainer готов. Следующая ячейка запускает обучение.')

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.27it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.27it/s]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 5/5 [00:00<00:00, 38.42it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 4/4 [00:00<00:00,  7.30it/s]


Trainer готов. Следующая ячейка запускает обучение.


/home/edu/amsafin/HSE-Avito/LLM/hw5/.venv/lib/python3.12/site-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1108 11:45:01.874566616 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


### Обучение и validation

Это прямой trainer.train() в текущем kernel. Progress bar показывает optimizer-шаги и останавливается на 120. Подготовка группы генераций может занимать десятки секунд до первого обновления счётчика. Когда счётчик дойдёт до 120, внутри того же вызова начнётся validation.

In [17]:
initial_parameters = {}
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        initial_parameters[name] = parameter.detach().float().cpu().clone()
torch.cuda.synchronize()
start = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
train_seconds = time.perf_counter() - start
print('Обучение и validation завершены, секунд:', round(train_seconds, 1))
pprint(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss
1,-0.011023
2,-0.252658
3,0.000000
4,0.168780
5,-0.084562
6,-0.352772
7,-0.051949
8,0.000000
9,0.000000
10,0.000000


Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 16, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 16, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 16, num_warps: 16, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 32, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 32, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 32, num_warps: 16, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 64, num_warps: 4, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel layer_norm_gated_bwd_kernel with config BT: 64, num_warps: 8, num_ctas: 1, num_stages: 3, maxnreg: None
Autotuning kernel laye

### Сохранить результаты и полную модель

Сначала сохраняем адаптер, историю, summary и validation-метрики. Затем объединяем LoRA с уже загруженной моделью и сохраняем полные веса для evaluate.py. После этого освобождаем память для следующего запуска.

In [18]:
print('Сохраняем адаптер и метрики...', flush=True)
if 'train_seconds' not in globals():
    train_seconds = None
summary = save_training_results(
    trainer=trainer,
    output_dir=output_dir,
    initial_parameters=initial_parameters,
    train_seconds=train_seconds,
    gpu_memory_before=gpu_memory_before,
)
pprint(summary)
print('Сохраняем объединённую модель:', model_dir, flush=True)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
del trainer, model, merged_model, initial_parameters
gc.collect()
torch.cuda.empty_cache()
print('Готово:', run_name)
print('Осталось выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Сохраняем адаптер и метрики...
{'changed_parameter_tensors': 24,
 'epoch': 0.01016260162601626,
 'finite_training_metrics': True,
 'generated_completions': 960,
 'gpu': 'NVIDIA A100-SXM4-80GB',
 'gpu_free_before_load_gib': 50.926513671875,
 'gpu_total_gib': 79.251708984375,
 'groups_with_reward_variation': 66,
 'optimizer_steps': 120,
 'parameter_delta_l2': 0.15652456128308379,
 'peak_allocated_gib': 73.76976203918457,
 'peak_reserved_gib': 76.955078125,
 'reward_groups': 120,
 'status': 'updated',
 'train_seconds': 335.0826256519649,
 'trainable_parameters': 417792,
 'truncated_completions': 85}
Сохраняем объединённую модель: checkpoints/rl_shaped/final


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.49s/it]


Готово: rl_shaped
Осталось выделено GPU-памяти, GiB: 48.15676689147949


## RL-3: меньший learning rate

Меняем только learning rate относительно RL-2. Гипотеза: меньший шаг уменьшит нежелательные изменения политики.

Выполни все ячейки этого блока, включая сохранение и освобождение памяти, прежде чем переходить к следующему эксперименту. Выходные папки должны быть пустыми: предыдущие результаты не перезаписываются.

In [7]:
run_name = 'rl_lr'
learning_rate = 1e-05
reward_weights = {'exact': 1.0, 'functions': 0.2, 'format': 0.05}
output_dir = Path('results') / run_name
model_dir = Path('checkpoints') / run_name / 'final'
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(f'Результаты уже существуют: {output_dir}')
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(42)
print('Эксперимент:', run_name)
print('Learning rate:', learning_rate)
print('Награда:', reward_weights)

Эксперимент: rl_lr
Learning rate: 1e-05
Награда: {'exact': 1.0, 'functions': 0.2, 'format': 0.05}


In [8]:
os.environ["TRITON_PRINT_AUTOTUNING"] = "0"

In [9]:
# del initial_parameters не помогает какая-то бага vllm держит память
# gc.collect()
# torch.cuda.empty_cache()

### Загрузить исходную модель и подключить LoRA

Веса каждый раз начинаются с одной и той же исходной ревизии. Ниже видны LoRA-параметры, число обучаемых весов и фактически занятая GPU-память.

In [10]:
print('Загрузка исходной Qwen...', flush=True)
gpu_memory_before = torch.cuda.mem_get_info()
torch.cuda.reset_peak_memory_stats()
model = load_base_model(language_only=False).to('cuda')
lora_config = LoraConfig(
    task_type='CAUSAL_LM',
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Загрузка исходной Qwen...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 617/617 [00:00<00:00, 7626.79it/s]


trainable params: 417,792 || all params: 2,213,659,456 || trainable%: 0.0189
Выделено GPU-памяти, GiB: 4.1240620613098145


### Создать Trainer

Функция награды получает реальные генерации, разбирает их и сравнивает с отдельным эталоном. Все генерации и компоненты награды сохраняются в results/<имя>/. Training и validation пишутся в разные файлы.

In [11]:
eos_token_ids = model.generation_config.eos_token_id or []
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
eos_token_ids = set(eos_token_ids) | {tokenizer.eos_token_id}
training_reward = make_training_reward(
    output_dir=output_dir,
    weights=reward_weights,
    eos_token_ids=eos_token_ids,
    max_completion_length=COMMON_GRPO['max_completion_length'],
    num_generations=COMMON_GRPO['num_generations'],
)
training_args = GRPOConfig(
    output_dir=str(output_dir),
    learning_rate=learning_rate,
    **COMMON_GRPO,
)
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=training_reward,
    train_dataset=train_dataset,
    eval_dataset=None,
    processing_class=tokenizer,
)
settings = {
    **COMMON_GRPO,
    'learning_rate': learning_rate,
    'reward_weights': reward_weights,
    'model': MODEL_ID,
    'revision': MODEL_REVISION,
    'max_prompt_tokens': MAX_PROMPT_TOKENS,
    'lora_rank': 8,
    'lora_alpha': 16,
    'target_modules': ['q_proj', 'v_proj'],
    'training_examples': len(train_dataset),
    'validation_examples': 0,
    'skipped_long_prompts': skipped_train,
    'skipped_long_validation_prompts': skipped_validation,
    'checkpoint_selection': 'optimizer step 120',
    'versions': {name: version(name) for name in ['torch', 'transformers', 'trl', 'peft', 'datasets']},
}
(output_dir / 'config.json').write_text(json.dumps(settings, indent=2))
print('Trainer готов. Следующая ячейка запускает обучение.')

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.36it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.36it/s]

Ca

Trainer готов. Следующая ячейка запускает обучение.


### Обучение и validation

Это прямой trainer.train() в текущем kernel. Progress bar показывает optimizer-шаги и останавливается на 120. Подготовка группы генераций может занимать десятки секунд до первого обновления счётчика. Когда счётчик дойдёт до 120, внутри того же вызова начнётся validation.

In [12]:
initial_parameters = {}
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        initial_parameters[name] = parameter.detach().float().cpu().clone()
torch.cuda.synchronize()
start = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
train_seconds = time.perf_counter() - start
print('Обучение и validation завершены, секунд:', round(train_seconds, 1))
pprint(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss
1,-0.011023
2,-0.260698
3,0.000000
4,-0.003797
5,0.032732
6,-0.444038
7,-0.148254
8,0.068733
9,-0.068657
10,0.000000


Обучение и validation завершены, секунд: 1645.2
{'epoch': 0.01016260162601626,
 'total_flos': 0.0,
 'train_loss': -0.03317774354169766,
 'train_runtime': 1645.1523,
 'train_samples_per_second': 0.584,
 'train_steps_per_second': 0.073}


### Сохранить результаты и полную модель

Сначала сохраняем адаптер, историю, summary и validation-метрики. Затем объединяем LoRA с уже загруженной моделью и сохраняем полные веса для evaluate.py. После этого освобождаем память для следующего запуска.

In [13]:
print('Сохраняем адаптер и метрики...', flush=True)
if 'train_seconds' not in globals():
    train_seconds = None
summary = save_training_results(
    trainer=trainer,
    output_dir=output_dir,
    initial_parameters=initial_parameters,
    train_seconds=train_seconds,
    gpu_memory_before=gpu_memory_before,
)
pprint(summary)
print('Сохраняем объединённую модель:', model_dir, flush=True)
merged_model = model.merge_and_unload()
merged_model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
del trainer, model, merged_model, initial_parameters
gc.collect()
torch.cuda.empty_cache()
print('Готово:', run_name)
print('Осталось выделено GPU-памяти, GiB:', torch.cuda.memory_allocated() / 1024**3)

Сохраняем адаптер и метрики...
{'changed_parameter_tensors': 24,
 'epoch': 0.01016260162601626,
 'finite_training_metrics': True,
 'generated_completions': 960,
 'gpu': 'NVIDIA A100-SXM4-80GB',
 'gpu_free_before_load_gib': 78.838623046875,
 'gpu_total_gib': 79.251708984375,
 'groups_with_reward_variation': 67,
 'optimizer_steps': 120,
 'parameter_delta_l2': 0.11085605642258818,
 'peak_allocated_gib': 51.54941368103027,
 'peak_reserved_gib': 78.107421875,
 'reward_groups': 120,
 'status': 'updated',
 'train_seconds': 1645.181967894081,
 'trainable_parameters': 417792,
 'truncated_completions': 89}
Сохраняем объединённую модель: checkpoints/rl_lr/final


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.94s/it]


Готово: rl_lr
Осталось выделено GPU-памяти, GiB: 26.13591957092285


## BFCL после трёх запусков

В обычном терминале dev-машины из hw5 выполни:

```bash
GPU=1 bash evaluate_all.sh
```

Это отдельный Docker-запуск исходного evaluate.py для baseline и трёх сохранённых моделей. Скрипт собирает метрики и сжатые генерации в artifacts/. Параметры и код evaluator не меняются. На сервере остаются полные веса checkpoints/ и адаптеры results/.

Для Git нужны artifacts/, notebook и код. Завтра локально analysis.ipynb прочитает эти результаты без GPU и сохранит таблицу и графики. Здесь нет subprocess, argparse или чтения configs/rl_*.json.

> докера на тачке не оказалось, <br>
.venv/bin/python -m pip install "soundfile==0.14.0" "numpy==2.2.6"
CUDA_VISIBLE_DEVICES=0 .venv/bin/python evaluate.py Qwen/Qwen3.5-2B --output artifacts/baseline